# Merging and Concatenating Data

In the explainer, we saw how organisational data ends up scattered across systems that were never designed to talk to each other. Different identifiers, different formats, different assumptions. We also saw that combining those sources without care leads to silent failures: mismatched keys, duplicated rows, inflated totals.

This notebook puts that into practice. We will work with shipment data from a global logistics company whose COO wants a single, unified view of all shipments worldwide. The data lives in three regional systems, each with its own format:

- **Asia**: a clean CSV with YYYY-MM-DD dates
- **Europe**: a CSV with different column names and DD/MM/YYYY dates
- **Americas**: a nested JSON file with Unix timestamps

This is a realistic extraction and loading problem. Each system was built independently, so nothing lines up by default. Our job is to make it line up.

By the end of this notebook we will be able to:

- Load data from CSV, JSON, and SQLite into pandas DataFrames
- Normalise column names and date formats across datasets
- Stack DataFrames vertically with `pd.concat`
- Join DataFrames horizontally with `pd.merge`
- Validate each operation by checking row counts, nulls, and duplicates

In [ ]:
%pip install -q pandas

In [ ]:
import pandas as pd
import json
import sqlite3
from datetime import datetime

---

## 1. Loading the Asia shipments (CSV)

The Asian regional system exports a straightforward CSV. This is the easiest source, the one that needs the least work. Starting here gives us a clean reference point to compare the other two against.

`pd.read_csv` reads the file into a **DataFrame**, pandas' core data structure: a table with labelled rows and columns.

In [ ]:
asia = pd.read_csv("../data/shipments_asia.csv")
print(f"Shape: {asia.shape}")
asia.head()

In [ ]:
# Quick validation: check for nulls and data types
print(asia.dtypes)
print()
print(asia.isnull().sum())

The Asia data is clean: no nulls, sensible types. The `ship_date` column is a string (`object`) rather than a proper datetime. We will fix that later when we combine the datasets.

We will also add a `region` column so that once the data is stacked, we can still tell where each row came from. This is a small thing, but it matters: after concatenation, the origin of each row is invisible unless we preserve it explicitly.

In [ ]:
asia["region"] = "Asia"
asia.head(3)

---

## 2. Loading the Europe shipments (CSV with differences)

Here is where the problems from the explainer start to appear. The European system uses different column names and a different date format. Compare its header row to Asia's:

| Asia column | Europe column |
|-------------|---------------|
| `shipment_id` | `id` |
| `cost_usd` | `cost_eur` |
| `ship_date` (YYYY-MM-DD) | `ship_date` (DD/MM/YYYY) |

If we tried to stack these two DataFrames right now, pandas would treat `shipment_id` and `id` as separate columns. The dates would end up in the same column but mean different things ("01/02/2024" would be ambiguous). We need to **rename columns** so they match and **convert the date format** before we can combine.

In [ ]:
europe = pd.read_csv("../data/shipments_europe.csv")
print(f"Shape: {europe.shape}")
print(f"Columns: {list(europe.columns)}")
europe.head()

### Renaming columns

The `.rename()` method takes a dictionary mapping old names to new names. We rename `id` to `shipment_id` and `cost_eur` to `cost_local` (we will standardise currency later).

In [ ]:
europe = europe.rename(columns={"id": "shipment_id", "cost_eur": "cost_local"})
print(list(europe.columns))

### Converting the date format

Europe's dates are in DD/MM/YYYY format. Remember the explainer's point about date formats: "15/03/2024" and "2024-03-15" represent the same day, but a computer sees two different strings. We use `pd.to_datetime` with an explicit `format` parameter to parse them correctly, then convert back to the YYYY-MM-DD string format used by Asia. The `dayfirst=True` parameter would also work, but specifying the exact format is safer because it fails loudly if the data doesn't match.

In [ ]:
# Before: DD/MM/YYYY strings
print("Before:", europe["ship_date"].iloc[0])

europe["ship_date"] = pd.to_datetime(europe["ship_date"], format="%d/%m/%Y").dt.strftime("%Y-%m-%d")

# After: YYYY-MM-DD strings
print("After: ", europe["ship_date"].iloc[0])

We also need to handle the currency difference. The European costs are in EUR, but our target schema uses USD. In a real project we would apply a proper exchange rate (or better, store both and convert at analysis time). For this exercise we will apply a rough conversion and note the simplification.

In [ ]:
# Approximate EUR to USD conversion (for demonstration purposes)
eur_to_usd = 1.08
europe["cost_usd"] = (europe["cost_local"] * eur_to_usd).round(2)
europe = europe.drop(columns=["cost_local"])

europe["region"] = "Europe"
europe.head(3)

---

## 3. Loading the Americas shipments (nested JSON)

The Americas system exports data as **nested JSON**, the format the explainer introduced as common in web systems and APIs. Each record has sub-objects for `route`, `details`, and `financials`, rather than flat columns. Before we can work with this data alongside the CSVs, we need to flatten the nested structure into rows and columns.

There are two approaches:
1. Use `pd.json_normalize` to flatten automatically
2. Parse manually with a list comprehension

We will use `pd.json_normalize` because it handles nested dictionaries cleanly.

In [ ]:
with open("../data/shipments_americas.json") as f:
    americas_raw = json.load(f)

# Inspect one record to see the nested structure
americas_raw[0]

In [ ]:
# json_normalize flattens nested dicts into columns with dot-separated names
americas = pd.json_normalize(americas_raw)
print(f"Shape: {americas.shape}")
print(f"Columns: {list(americas.columns)}")
americas.head(3)

The columns now have names like `route.origin` and `details.weight_kg`. These dot-separated names come from the JSON nesting. We need to rename them to match our target schema and convert the Unix timestamps to YYYY-MM-DD date strings. That is a third date format from a third system, exactly the kind of inconsistency we would expect.

In [ ]:
americas = americas.rename(columns={
    "route.origin": "origin",
    "route.destination": "destination",
    "details.weight_kg": "weight_kg",
    "details.ship_date_unix": "ship_date_unix",
    "financials.cost_usd": "cost_usd",
    "financials.insurance_usd": "insurance_usd",
})

# Convert Unix timestamps to YYYY-MM-DD strings
americas["ship_date"] = pd.to_datetime(americas["ship_date_unix"], unit="s").dt.strftime("%Y-%m-%d")
americas = americas.drop(columns=["ship_date_unix"])

americas["region"] = "Americas"
americas.head(3)

---

## 4. Concatenating with `pd.concat`

We now have three DataFrames that describe the same kind of thing (shipments) but came from different systems. **`pd.concat`** stacks them vertically, one on top of the other, to produce a single unified table.

Before concatenating, the DataFrames need to share the same column names for the columns that represent the same data. Let us check what we have.

In [ ]:
print("Asia columns:    ", sorted(asia.columns))
print("Europe columns:  ", sorted(europe.columns))
print("Americas columns:", sorted(americas.columns))

The columns are not identical. Asia has `supplier_id`, and Americas has `insurance_usd`. That is fine. `pd.concat` fills missing columns with `NaN` by default, which is exactly the right behaviour: it means "this source didn't have this information." We use `ignore_index=True` to reset the row numbers so they run 0, 1, 2, ... instead of restarting for each source.

In [ ]:
shipments = pd.concat([asia, europe, americas], ignore_index=True)
print(f"Combined shape: {shipments.shape}")
print(f"Expected rows:  {len(asia) + len(europe) + len(americas)}")
shipments.head()

### Validation after concatenation

The explainer stressed that combinations can fail silently. A good habit is to validate immediately after every concat or merge. The row count of the result should equal the sum of the input row counts. If it doesn't, something went wrong. The null counts tell us which columns have gaps and whether those gaps are expected.

In [ ]:
# Row count check
assert len(shipments) == len(asia) + len(europe) + len(americas), "Row count mismatch!"

# Null check
print(shipments.isnull().sum())
print()

# Rows per region
print(shipments["region"].value_counts())

The nulls are expected: `supplier_id` only exists for Asia, and `insurance_usd` only exists for Americas. The core columns (`shipment_id`, `origin`, `destination`, `weight_kg`, `cost_usd`, `ship_date`, `carrier`, `status`, `region`) are complete.

---

## 5. Merging with supplier data (SQLite)

Concatenation stacked rows from similar sources. Now we need a different operation: **merging**, which joins columns from different tables based on a shared key. This is the horizontal join the explainer discussed, matching rows across datasets using a common identifier.

The COO wants supplier information alongside each Asian shipment. The supplier details live in a SQLite database. We use `pd.read_sql` to load the table, then `pd.merge` to join it onto our combined dataset.

**`pd.merge`** performs a SQL-style join. The `on` parameter specifies the join column (the shared key), and `how` controls the join type (inner, left, right, outer).

In [ ]:
conn = sqlite3.connect("../data/suppliers.sqlite")
suppliers = pd.read_sql("SELECT * FROM suppliers", conn)
conn.close()

print(f"Suppliers shape: {suppliers.shape}")
suppliers.head()

In [ ]:
# Left join: keep all shipments, attach supplier info where available
# Rows without a supplier_id (Europe, Americas) will get NaN for supplier columns
shipments = pd.merge(
    shipments,
    suppliers,
    on="supplier_id",
    how="left",
    suffixes=("", "_supplier"),
)

print(f"Shape after merge: {shipments.shape}")
shipments.head()

### Validation after merge

A left join should not change the row count. Every row from the left DataFrame is preserved, whether or not it found a match. If the row count increases, it means the join key had duplicates on the right side, producing the kind of row multiplication the explainer warned about. Always check.

In [ ]:
expected_rows = len(asia) + len(europe) + len(americas)
print(f"Rows before merge: {expected_rows}")
print(f"Rows after merge:  {len(shipments)}")
assert len(shipments) == expected_rows, "Row count changed after merge!"

# Check that supplier info was attached to Asian shipments
asia_with_supplier = shipments[shipments["region"] == "Asia"]["name"].notna().sum()
print(f"Asian shipments with supplier name: {asia_with_supplier}/{len(asia)}")

# Confirm non-Asian rows have NaN for supplier columns
europe_supplier_nulls = shipments[shipments["region"] == "Europe"]["name"].isna().all()
print(f"All European rows have null supplier name: {europe_supplier_nulls}")

---

## 6. Summary of join types

| Join type | `how=` | Keeps |
|-----------|--------|-------|
| Inner | `"inner"` | Only rows where the key exists in both DataFrames |
| Left | `"left"` | All rows from the left DataFrame, matched rows from the right |
| Right | `"right"` | All rows from the right DataFrame, matched rows from the left |
| Outer | `"outer"` | All rows from both DataFrames |

We used a left join because we wanted to keep every shipment, whether or not it had a matching supplier. An inner join would have silently dropped all European and Americas shipments — a serious loss of data that might not be obvious at first glance.

---

## Exercises

Now it is your turn. Complete each exercise in the code cell provided.

### Exercise 1: Load and flatten the Americas JSON manually

Instead of using `pd.json_normalize`, write a list comprehension that extracts the flat fields from each record in `americas_raw`. Create a DataFrame from it with columns: `shipment_id`, `origin`, `destination`, `weight_kg`, `cost_usd`, `carrier`, `status`. Print the shape and first 3 rows.

In [ ]:
# Your code here


### Exercise 2: Concatenate two CSVs and check for column alignment

Load `shipments_asia.csv` and `shipments_europe.csv` fresh. Without renaming any columns, concatenate them using `pd.concat`. Inspect the result: which columns have `NaN` values, and why? Print the null counts per column.

In [ ]:
# Your code here


### Exercise 3: Inner join vs left join

Using the combined `shipments` DataFrame (before the supplier merge) and the `suppliers` DataFrame:

1. Perform an **inner join** on `supplier_id`. How many rows does the result have? Why?
2. Perform a **left join** on `supplier_id`. How many rows does the result have? Why is it different?

Print both row counts and write a brief comment explaining the difference.

In [ ]:
# Your code here


### Exercise 4: Build the full pipeline

Starting from scratch, write a single cell that:

1. Loads all three shipment files (Asia CSV, Europe CSV, Americas JSON)
2. Normalises column names and date formats
3. Adds a `region` column to each
4. Concatenates them into one DataFrame
5. Loads suppliers from SQLite and merges them with a left join
6. Prints the final shape, columns, and null counts

This is a complete data-loading pipeline. Aim for clean, readable code.

In [ ]:
# Your code here


---

## Summary

We have now completed the extraction and loading stages of a realistic data integration task. Starting from three separate systems with different formats, different column names, and different date conventions, we built a single combined dataset and validated it at every step.

In this notebook we:

- Loaded CSV data with `pd.read_csv` and inspected it with `.shape`, `.dtypes`, and `.isnull().sum()`
- Flattened nested JSON into a DataFrame with `pd.json_normalize`
- Renamed columns with `.rename()` to align different schemas
- Converted date formats using `pd.to_datetime` with explicit `format` parameters
- Stacked DataFrames vertically with **`pd.concat`** and used `ignore_index=True`
- Read a SQLite table into pandas with `pd.read_sql`
- Joined DataFrames horizontally with **`pd.merge`** using a left join
- Validated each step by checking row counts, null patterns, and assertions

The data is combined, but it is not yet clean. In the next notebook we will tackle the transform stage: handling nulls, removing duplicates, and normalising messy strings in the warehouse inventory.